# 🫀 VisionCare: Multi-Modal CVD Detection

## Complete Training Pipeline (Drive Version)

---

### ✅ Dataset Already Downloaded!
Your Symile-MIMIC data is in: `/content/drive/MyDrive/symile-mimic/`

---

### 📋 What This Notebook Does:
1. **Mounts** Google Drive
2. **Loads** data from Drive (instant!)
3. **Trains** individual + fusion models
4. **Compares** performance
5. **Exports** trained models

---

### 🔀 Fusion Strategy: INTERMEDIATE FUSION
```
CXR → ResNet-50 → 2048 features ─┐
ECG → 1D-CNN   →  256 features ──┼→ Concat (2368) → Fusion MLP → CVD Risk
Labs → MLP     →   64 features ──┘
```

---
# 1️⃣ Setup & Mount Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted!")
else:
    print("✅ Drive already mounted!")

# Set data directory
DATA_DIR = "/content/drive/MyDrive/symile-mimic"
print(f"\n📂 Data directory: {DATA_DIR}")

# Verify data exists
if os.path.exists(f"{DATA_DIR}/train.csv"):
    print("✅ Dataset found!")
    !ls -la {DATA_DIR}/*.csv
else:
    print("❌ Dataset not found! Run the AWS sync first.")

In [ ]:
# Check GPU & install packages
import torch
import sys

print("="*60)
print("🖥️ SYSTEM CHECK")
print("="*60)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    DEVICE = torch.device('cuda')
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")
    DEVICE = torch.device('cpu')

print("="*60)

# Install packages
!pip install -q tqdm scikit-learn matplotlib seaborn
print("\n✅ Packages ready!")

---
# 2️⃣ Load Dataset from Drive

Loading directly from Drive - no download needed! 🚀

In [ ]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

class SymileMIMICDataset(Dataset):
    """
    Symile-MIMIC Dataset for Multi-Modal CVD Detection.
    Loads from Google Drive.
    """
    
    def __init__(self, data_dir, split='train'):
        self.data_dir = data_dir
        self.split = split
        
        print(f"\n📂 Loading {split.upper()} split from Drive...")
        
        # Load CSV with metadata and labels
        self.df = pd.read_csv(f"{data_dir}/{split}.csv")
        
        # Load numpy arrays (memory-mapped)
        npy_dir = f"{data_dir}/data_npy/{split}"
        
        print(f"   Loading CXR...")
        self.cxr = np.load(f"{npy_dir}/cxr_{split}.npy", mmap_mode='r')
        
        print(f"   Loading ECG...")
        self.ecg = np.load(f"{npy_dir}/ecg_{split}.npy", mmap_mode='r')
        
        print(f"   Loading Labs...")
        self.labs_pct = np.load(f"{npy_dir}/labs_percentiles_{split}.npy", mmap_mode='r')
        self.labs_miss = np.load(f"{npy_dir}/labs_missingness_{split}.npy", mmap_mode='r')
        
        # Extract Cardiomegaly labels (our CVD target)
        if 'Cardiomegaly' in self.df.columns:
            labels = self.df['Cardiomegaly'].fillna(0).values
            self.labels = ((labels == 1.0) | (labels == -1.0)).astype(int)
        else:
            print("⚠️ Cardiomegaly column not found!")
            self.labels = np.zeros(len(self.df), dtype=int)
        
        # Print statistics
        print(f"   ✅ Loaded {len(self.df):,} samples")
        print(f"   CXR shape: {self.cxr.shape}")
        print(f"   ECG shape: {self.ecg.shape}")
        print(f"   Cardiomegaly+: {self.labels.sum():,} ({self.labels.mean()*100:.1f}%)")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # CXR: (3, 320, 320)
        cxr = torch.tensor(self.cxr[idx].copy(), dtype=torch.float32)
        
        # ECG: (1, 5000, 12) → (12, 5000)
        ecg = torch.tensor(self.ecg[idx].copy(), dtype=torch.float32)
        ecg = ecg.squeeze(0).transpose(0, 1)
        
        # Labs: (100,)
        labs = np.concatenate([self.labs_pct[idx], self.labs_miss[idx]])
        labs = torch.tensor(labs, dtype=torch.float32)
        
        return cxr, ecg, labs, int(self.labels[idx])


# Load datasets
print("="*60)
print("📊 LOADING DATASETS FROM DRIVE")
print("="*60)

train_dataset = SymileMIMICDataset(DATA_DIR, 'train')
val_dataset = SymileMIMICDataset(DATA_DIR, 'val')

# Create DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\n✅ DataLoaders ready!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print("="*60)

In [ ]:
# Visualize a sample
import matplotlib.pyplot as plt

sample_cxr, sample_ecg, sample_labs, sample_labels = next(iter(train_loader))

print("📦 Sample Batch Shapes:")
print(f"   CXR:  {sample_cxr.shape}")
print(f"   ECG:  {sample_ecg.shape}")
print(f"   Labs: {sample_labs.shape}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CXR
img = sample_cxr[0].permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min())
axes[0].imshow(img)
axes[0].set_title(f"🩻 CXR (Label: {sample_labels[0].item()})")
axes[0].axis('off')

# ECG
for lead in range(3):
    axes[1].plot(sample_ecg[0, lead, :500].numpy(), label=f"Lead {lead+1}")
axes[1].set_title("❤️ ECG (3 leads, 500 samples)")
axes[1].legend()

# Labs
axes[2].bar(range(50), sample_labs[0, :50].numpy(), alpha=0.7)
axes[2].set_title("🩸 Blood Labs (50 tests)")

plt.tight_layout()
plt.show()

---
# 3️⃣ Define Models

In [ ]:
from torchvision import models

# Vision Module (ResNet-50)
class VisionModule(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.name = "ResNet-50"
        self.feature_dim = 2048
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features), features


# Signal Module (1D-CNN)
class SignalModule(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.name = "1D-CNN"
        self.feature_dim = 256
        self.conv = nn.Sequential(
            nn.Conv1d(12, 64, 15, padding=7), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(64, 128, 11, padding=5), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(128, 256, 7, padding=3), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Linear(256, num_classes)
    
    def forward(self, x):
        features = self.conv(x).squeeze(-1)
        return self.classifier(features), features


# Clinical Module (MLP)
class ClinicalModule(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.name = "MLP"
        self.feature_dim = 64
        self.encoder = nn.Sequential(
            nn.Linear(100, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.classifier = nn.Linear(64, num_classes)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features), features


# Fusion Module
class VisionCareFusion(nn.Module):
    def __init__(self, vision, signal, clinical, num_classes=2):
        super().__init__()
        self.vision = vision
        self.signal = signal
        self.clinical = clinical
        
        total = vision.feature_dim + signal.feature_dim + clinical.feature_dim
        self.fusion = nn.Sequential(
            nn.Linear(total, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        print(f"🔀 Fusion: {total} features → 2 classes")
    
    def forward(self, cxr, ecg, labs):
        _, v = self.vision(cxr)
        _, s = self.signal(ecg)
        _, c = self.clinical(labs)
        combined = torch.cat([v, s, c], dim=1)
        return self.fusion(combined), (v, s, c)

print("✅ Models defined!")

---
# 4️⃣ Training Functions

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score
import time

def train_epoch(model, loader, optimizer, criterion, modality='fusion'):
    model.train()
    total_loss = 0
    
    for cxr, ecg, labs, y in tqdm(loader, desc="Training", leave=False):
        cxr, ecg, labs, y = cxr.to(DEVICE), ecg.to(DEVICE), labs.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        
        if modality == 'fusion':
            logits, _ = model(cxr, ecg, labs)
        elif modality == 'vision':
            logits, _ = model(cxr)
        elif modality == 'signal':
            logits, _ = model(ecg)
        elif modality == 'clinical':
            logits, _ = model(labs)
        
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluate(model, loader, modality='fusion'):
    model.eval()
    all_probs, all_labels = [], []
    
    with torch.no_grad():
        for cxr, ecg, labs, y in tqdm(loader, desc="Evaluating", leave=False):
            cxr, ecg, labs = cxr.to(DEVICE), ecg.to(DEVICE), labs.to(DEVICE)
            
            if modality == 'fusion':
                logits, _ = model(cxr, ecg, labs)
            elif modality == 'vision':
                logits, _ = model(cxr)
            elif modality == 'signal':
                logits, _ = model(ecg)
            elif modality == 'clinical':
                logits, _ = model(labs)
            
            probs = F.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.numpy())
    
    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_probs])
    return {'auc': auc, 'accuracy': acc}


def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, modality='fusion'):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
    criterion = nn.CrossEntropyLoss()
    
    best_auc = 0
    history = {'train_loss': [], 'val_auc': []}
    
    print(f"\n🚀 Training {modality.upper()}...")
    start = time.time()
    
    for epoch in range(epochs):
        loss = train_epoch(model, train_loader, optimizer, criterion, modality)
        metrics = evaluate(model, val_loader, modality)
        scheduler.step(metrics['auc'])
        
        history['train_loss'].append(loss)
        history['val_auc'].append(metrics['auc'])
        
        print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} | AUC: {metrics['auc']:.4f} | Acc: {metrics['accuracy']*100:.1f}%")
        
        if metrics['auc'] > best_auc:
            best_auc = metrics['auc']
            torch.save(model.state_dict(), f'best_{modality}_model.pth')
            print(f"         ✅ Saved!")
    
    print(f"⏱️ Time: {(time.time()-start)/60:.1f} min | Best AUC: {best_auc:.4f}")
    return history, best_auc

print("✅ Training functions ready!")

---
# 5️⃣ Train All Models 🚀

In [ ]:
# Configuration
EPOCHS = 10  # Increase to 20-50 for better results
LR = 1e-4

results = {}
print("="*60)
print("🎯 TRAINING ALL MODELS")
print("="*60)

In [ ]:
# 1. Vision (CXR only)
vision_model = VisionModule()
h_v, auc_v = train_model(vision_model, train_loader, val_loader, EPOCHS, LR, 'vision')
results['vision'] = {'auc': auc_v, 'history': h_v}

In [ ]:
# 2. Signal (ECG only)
signal_model = SignalModule()
h_s, auc_s = train_model(signal_model, train_loader, val_loader, EPOCHS, LR, 'signal')
results['signal'] = {'auc': auc_s, 'history': h_s}

In [ ]:
# 3. Clinical (Labs only)
clinical_model = ClinicalModule()
h_c, auc_c = train_model(clinical_model, train_loader, val_loader, EPOCHS, LR, 'clinical')
results['clinical'] = {'auc': auc_c, 'history': h_c}

In [ ]:
# 4. FUSION (All modalities) - The main model!
v = VisionModule()
s = SignalModule()
c = ClinicalModule()
fusion_model = VisionCareFusion(v, s, c)

h_f, auc_f = train_model(fusion_model, train_loader, val_loader, EPOCHS, LR, 'fusion')
results['fusion'] = {'auc': auc_f, 'history': h_f}

---
# 6️⃣ Compare Results 📊

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
names = ['Vision\n(CXR)', 'Signal\n(ECG)', 'Clinical\n(Labs)', 'FUSION\n(All)']
aucs = [results['vision']['auc'], results['signal']['auc'], results['clinical']['auc'], results['fusion']['auc']]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

bars = axes[0].bar(names, aucs, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('🏆 Model Comparison', fontsize=14)
axes[0].set_ylim(0.5, 1.0)
for bar, auc in zip(bars, aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{auc:.3f}', ha='center', fontweight='bold')

# Training curves
for name, color in zip(['vision', 'signal', 'clinical', 'fusion'], colors):
    axes[1].plot(results[name]['history']['val_auc'], label=name.capitalize(), color=color, linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation AUC')
axes[1].set_title('📈 Training Progress')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visioncare_results.png', dpi=150)
plt.show()

# Summary
print("\n" + "="*50)
print("📊 FINAL RESULTS")
print("="*50)
for name in ['vision', 'signal', 'clinical', 'fusion']:
    star = '🥇' if name == 'fusion' else '  '
    print(f"{star} {name.capitalize():<12}: {results[name]['auc']:.4f}")

best_single = max(aucs[:3])
improvement = (results['fusion']['auc'] - best_single) / best_single * 100
print(f"\n🚀 Fusion improvement: {improvement:+.1f}%")

---
# 7️⃣ Save & Download Models

In [ ]:
import json

# Save config
config = {
    'modalities': {'vision': 2048, 'signal': 256, 'clinical': 64},
    'fusion': 'intermediate',
    'results': {k: v['auc'] for k, v in results.items()},
    'input_shapes': {'cxr': [3, 320, 320], 'ecg': [12, 5000], 'labs': [100]}
}

with open('visioncare_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Copy to Drive for persistence
!cp best_fusion_model.pth /content/drive/MyDrive/symile-mimic/
!cp visioncare_config.json /content/drive/MyDrive/symile-mimic/
!cp visioncare_results.png /content/drive/MyDrive/symile-mimic/

print("✅ Saved files:")
print("   - best_fusion_model.pth")
print("   - visioncare_config.json")
print("   - visioncare_results.png")
print("\n📂 Also copied to Drive for persistence!")

In [ ]:
# Download to computer
from google.colab import files

print("📥 Downloading...")
files.download('best_fusion_model.pth')
files.download('visioncare_config.json')
files.download('visioncare_results.png')

print("\n🎉 DONE! Your VisionCare model is ready for the dashboard!")

---
# 🎉 Training Complete!

## Files Generated:
- `best_fusion_model.pth` - Main trained model
- `visioncare_config.json` - Model configuration
- `visioncare_results.png` - Comparison chart

## Next Steps:
1. Download files to your computer
2. Create FastAPI backend for inference
3. Build React dashboard
4. Deploy!

---
### 🫀 VisionCare - BTP Semester 7